# Midterm — Green Rectangle Detection

**Course:** CSCI 450 — Machine Learning  
**Author:** Malek Elaghel  
**Original submission:** Fall 2023

## Topics covered

A small image-processing midterm question: locate the bounding box
of the green rectangle in a test image (`test.jpg`) using only NumPy
and `skimage.io`. The mask is built per-pixel: a pixel is "green" if
its green channel exceeds *both* its red and blue channels.

## How to run

The notebook expects a file called `test.jpg` next to it. The file
itself isn't included in the repo — pre-generated outputs in the
saved cells show the expected answer (top-left ≈ (143, 79),
bottom-right ≈ (176, 144)).

> The original kernel crashed while displaying the full
> `green_pixels` tuple. That last cell is left as-is (it just shows
> the raw arrays), and the reflection cell below explains a cleaner
> alternative.

---


In [2]:
import numpy as np
from skimage import io
    
image = io.imread('test.jpg')

# Find the rows and columns that contain green pixelss
green_pixels = np.where((image[..., 1] > image[..., 0]) & (image[..., 1] > image[..., 2]))

min_row, min_col = np.min(green_pixels, axis=1)
max_row, max_col = np.max(green_pixels, axis=1)
print(f"Green Rectangle Location:\nTop-Left Corn er (row, col): ({min_row}, {min_col})\nBottom-Right Corner (row, col): ({max_row}, {max_col})")


Green Rectangle Location:
Top-Left Corner (row, col): (143, 79)
Bottom-Right Corner (row, col): (176, 144)


In [3]:
green_pixels


(array([143, 143, 143, ..., 176, 176, 176], dtype=int64),
 array([ 79,  80,  81, ..., 137, 138, 139], dtype=int64))

: 

---

## Reflection — what I'd do differently now

1. **The "green > red AND green > blue" rule is fragile.** Anti-
   aliased edges or off-pure-green colours can satisfy it even when
   the human eye doesn't read the pixel as green. A better approach
   is to convert to **HSV** and threshold on hue (e.g. 80°–160°)
   *and* require minimum saturation/value — far more robust to
   lighting.
2. **`np.where` on a mask returns a tuple of two 1-D arrays, not a
   2-D array.** The original code's `np.min(green_pixels, axis=1)`
   silently does the right thing because tuples are iterable, but it
   would be clearer to write
   `rows, cols = np.nonzero(mask); min_row, min_col = rows.min(), cols.min()`.
3. **No connected-component analysis.** If the image had multiple
   green objects, the bounding box would span all of them. Using
   `skimage.measure.label` + `regionprops` would let us return one
   bounding box *per* green region.
4. **Print the full last cell killed the kernel** because the array
   was huge. A `green_pixels[0][:5]` or `pd.DataFrame(...).head()`
   call would have been safer for inspection.